<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/07_Modern_Architecture_Variants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Notebook 07: Modern Architecture Variants (LLaMA-style)
# Cell 1 — Setup

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# Shared config — same backbone as notebooks 04–06
cfg = {
    "vocab_size": 50257,
    "d_model":    512,
    "num_heads":  8,
    "n_kv_heads": 2,      # NEW (GQA): 8 query heads share 2 K/V heads -> 4x smaller KV cache
    "d_ff":       2048,   # standard FFN hidden; SwiGLU will derive its own width from this
    "num_layers": 6,
    "max_len":    256,
    "dropout":    0.0,
}

assert cfg["num_heads"] % cfg["n_kv_heads"] == 0, "num_heads must be divisible by n_kv_heads"

print(f"Device: {device}")
print(f"Query heads: {cfg['num_heads']}  |  KV heads: {cfg['n_kv_heads']}  "
      f"|  group size: {cfg['num_heads'] // cfg['n_kv_heads']}")
print(f"Config: {cfg}")

Device: cuda
Query heads: 8  |  KV heads: 2  |  group size: 4
Config: {'vocab_size': 50257, 'd_model': 512, 'num_heads': 8, 'n_kv_heads': 2, 'd_ff': 2048, 'num_layers': 6, 'max_len': 256, 'dropout': 0.0}


In [ ]:
# Cell 2 — RMSNorm

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # learnable gain, no bias

    def forward(self, x):
        dtype = x.dtype
        x = x.float()  # reduce in fp32 for stability, then cast back
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (self.weight * (x / rms)).to(dtype)


d = cfg["d_model"]
rms = RMSNorm(d).to(device)
ln_noaffine = nn.LayerNorm(d, elementwise_affine=False, eps=1e-5).to(device)

torch.manual_seed(0)
x = torch.randn(4, 16, d, device=device)

# Case A: zero-mean input -> should match biasless LayerNorm
x0 = x - x.mean(dim=-1, keepdim=True)
diff_zero_mean = (rms(x0) - ln_noaffine(x0)).abs().max().item()

# Case B: offset input -> should diverge (RMSNorm doesn't re-center)
x_off = x + 3.0
diff_offset = (rms(x_off) - ln_noaffine(x_off)).abs().max().item()

print(f"Max |RMSNorm - LayerNorm|, zero-mean input: {diff_zero_mean:.2e}")
print(f"Max |RMSNorm - LayerNorm|, offset input:    {diff_offset:.2e}")
print(f"RMSNorm params for d={d}: {sum(p.numel() for p in rms.parameters())} (gain only)")

Max |RMSNorm - LayerNorm|, zero-mean input: 4.77e-07
Max |RMSNorm - LayerNorm|, offset input:    3.78e+00
RMSNorm params for d=512: 512 (gain only)


In [ ]:
# Cell 3 — SwiGLU

class GELU_FFN(nn.Module):
    """Your notebooks 04-06 FFN: two matrices, single activation. No bias for a clean compare."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class SwiGLU(nn.Module):
    """LLaMA FFN: gate, up, down = three matrices. Swish gate modulates the value path."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)   # -> Swish
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)   # value path
        self.w_down = nn.Linear(d_ff, d_model, bias=False)   # back to d_model
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))   # silu == Swish (β=1)


def n_params(m):
    return sum(p.numel() for p in m.parameters())

d_model, d_ff_std = cfg["d_model"], cfg["d_ff"]

# The 2/3 trick: match params by solving 3 * d_model * h = 2 * d_model * d_ff_std
d_ff_swiglu = round((2/3) * d_ff_std)

ffn_gelu        = GELU_FFN(d_model, d_ff_std).to(device)
swiglu_naive    = SwiGLU(d_model, d_ff_std).to(device)      # same width -> too big
swiglu_matched  = SwiGLU(d_model, d_ff_swiglu).to(device)   # 2/3 width -> matched

print(f"GELU FFN      (d_ff={d_ff_std}):  {n_params(ffn_gelu):,} params")
print(f"SwiGLU naive  (d_ff={d_ff_std}):  {n_params(swiglu_naive):,} params  "
      f"({n_params(swiglu_naive)/n_params(ffn_gelu):.2f}x)")
print(f"SwiGLU 2/3    (d_ff={d_ff_swiglu}):  {n_params(swiglu_matched):,} params  "
      f"({n_params(swiglu_matched)/n_params(ffn_gelu):.3f}x)")

# Shape sanity check
xb = torch.randn(2, 16, d_model, device=device)
print(f"\nForward OK — in {tuple(xb.shape)} -> out {tuple(swiglu_matched(xb).shape)}")

GELU FFN      (d_ff=2048):  2,097,152 params
SwiGLU naive  (d_ff=2048):  3,145,728 params  (1.50x)
SwiGLU 2/3    (d_ff=1365):  2,096,640 params  (1.000x)

Forward OK — in (2, 16, 512) -> out (2, 16, 512)


In [ ]:
# Cell 4 — RoPE (Rotary Position Embedding)

head_dim = cfg["d_model"] // cfg["num_heads"]   # 64
assert head_dim % 2 == 0

def build_rope_cache(dim, max_len, base=10000.0, device=None):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))  # (dim/2,)
    t = torch.arange(max_len, device=device).float()                                   # (max_len,)
    freqs = torch.outer(t, inv_freq)            # (max_len, dim/2): angle = position * freq
    emb = torch.cat([freqs, freqs], dim=-1)     # (max_len, dim): duplicate for rotate_half
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):                    # x: (..., seq, dim); cos/sin: (seq, dim)
    return x * cos + rotate_half(x) * sin

cos, sin = build_rope_cache(head_dim, cfg["max_len"], device=device)

torch.manual_seed(1)
q = torch.randn(head_dim, device=device)
k = torch.randn(head_dim, device=device)

def score(qv, kv, m, n):                         # ⟨R(m)q, R(n)k⟩
    qm = apply_rope(qv.unsqueeze(0), cos[m:m+1], sin[m:m+1])
    kn = apply_rope(kv.unsqueeze(0), cos[n:n+1], sin[n:n+1])
    return (qm * kn).sum().item()

# (1) position 0 is the identity rotation
id_err = (apply_rope(q.unsqueeze(0), cos[0:1], sin[0:1]) - q).abs().max().item()
print(f"(1) RoPE at position 0 == identity:  max err {id_err:.2e}")

# (2) THE PROPERTY: four pairs, all with offset n-m = 5 -> all identical scores
print("\n(2) score depends only on relative offset (all offset = 5):")
for (m, n) in [(3, 8), (10, 15), (100, 105), (200, 205)]:
    print(f"    m={m:>3}, n={n:>3}:  score = {score(q, k, m, n):.6f}")

# (3) contrast: raw dot product is position-blind without RoPE
print(f"\n(3) raw q·k (no RoPE):  {(q*k).sum().item():.6f}   (constant, no position info)")

# (4) long-term decay: mean self-affinity ⟨R(0)q, R(d)q⟩/|q|² vs distance d
torch.manual_seed(2)
Q = torch.randn(256, head_dim, device=device)
q0 = apply_rope(Q, cos[0:1], sin[0:1])
norms = (Q * Q).sum(-1)
print("\n(4) mean normalized self-affinity vs distance d:")
for d in [0, 1, 2, 4, 8, 16, 32, 64, 128, 255]:
    qd  = apply_rope(Q, cos[d:d+1], sin[d:d+1])
    rel = ((q0 * qd).sum(-1) / norms).mean().item()
    print(f"    d={d:>3}:  {rel:+.3f}")

(1) RoPE at position 0 == identity:  max err 0.00e+00

(2) score depends only on relative offset (all offset = 5):
    m=  3, n=  8:  score = 10.477760
    m= 10, n= 15:  score = 10.477761
    m=100, n=105:  score = 10.477757
    m=200, n=205:  score = 10.477764

(3) raw q·k (no RoPE):  4.707331   (constant, no position info)

(4) mean normalized self-affinity vs distance d:
    d=  0:  +1.000
    d=  1:  +0.968
    d=  2:  +0.890
    d=  4:  +0.759
    d=  8:  +0.710
    d= 16:  +0.613
    d= 32:  +0.611
    d= 64:  +0.444
    d=128:  +0.296
    d=255:  +0.329


In [ ]:
# Cell 5 — Grouped-Query Attention (with RoPE)

class GQAttention(nn.Module):
    def __init__(self, cfg, cos, sin):
        super().__init__()
        self.n_q   = cfg["num_heads"]          # 8 query heads
        self.n_kv  = cfg["n_kv_heads"]         # 2 K/V heads
        self.group = self.n_q // self.n_kv     # 4 query heads per K/V head
        self.hd    = cfg["d_model"] // self.n_q  # 64
        d = cfg["d_model"]

        # Q is full-width; K and V are NARROWER — only n_kv heads. This is the whole saving.
        self.w_q = nn.Linear(d, self.n_q  * self.hd, bias=False)
        self.w_k = nn.Linear(d, self.n_kv * self.hd, bias=False)
        self.w_v = nn.Linear(d, self.n_kv * self.hd, bias=False)
        self.w_o = nn.Linear(self.n_q * self.hd, d, bias=False)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.w_q(x).view(B, T, self.n_q,  self.hd).transpose(1, 2)  # (B, 8, T, 64)
        k = self.w_k(x).view(B, T, self.n_kv, self.hd).transpose(1, 2)  # (B, 2, T, 64)
        v = self.w_v(x).view(B, T, self.n_kv, self.hd).transpose(1, 2)  # (B, 2, T, 64)

        # RoPE on q and k (not v) — broadcasts over the head axis
        c, s = self.cos[:T], self.sin[:T]
        q = apply_rope(q, c, s)
        k = apply_rope(k, c, s)

        # GQA core: each K/V head serves `group` query heads -> expand 2 -> 8
        k = k.repeat_interleave(self.group, dim=1)   # (B, 8, T, 64)
        v = v.repeat_interleave(self.group, dim=1)   # (B, 8, T, 64)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # Flash backend on T4
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_q * self.hd)
        return self.w_o(out)


attn = GQAttention(cfg, cos, sin).to(device)
xb = torch.randn(2, 32, cfg["d_model"], device=device)
print(f"Forward OK — in {tuple(xb.shape)} -> out {tuple(attn(xb).shape)}")

# --- KV-cache arithmetic ---
def kv_cache_bytes(n_kv, seq, layers, hd, batch=1, dtype_bytes=2):  # fp16 = 2 bytes
    return 2 * batch * layers * seq * n_kv * hd * dtype_bytes        # 2 = K and V

L, hd = cfg["num_layers"], cfg["d_model"] // cfg["num_heads"]
print("\nKV-cache size (fp16, batch=1, all 6 layers):")
for seq in [256, 2048, 8192]:
    mha = kv_cache_bytes(cfg["num_heads"],   seq, L, hd)
    gqa = kv_cache_bytes(cfg["n_kv_heads"],  seq, L, hd)
    print(f"  seq={seq:>5}:  MHA {mha/1e6:7.2f} MB   GQA {gqa/1e6:6.2f} MB   "
          f"saving {mha/gqa:.0f}x ({(mha-gqa)/1e6:.2f} MB freed)")

# Param cost of the attention projections: GQA vs full MHA
mha_kv = 2 * cfg["d_model"] * (cfg["num_heads"]  * hd)
gqa_kv = 2 * cfg["d_model"] * (cfg["n_kv_heads"] * hd)
print(f"\nK+V projection params:  MHA {mha_kv:,}   GQA {gqa_kv:,}   "
      f"(saved {mha_kv-gqa_kv:,})")

Forward OK — in (2, 32, 512) -> out (2, 32, 512)

KV-cache size (fp16, batch=1, all 6 layers):
  seq=  256:  MHA    3.15 MB   GQA   0.79 MB   saving 4x (2.36 MB freed)
  seq= 2048:  MHA   25.17 MB   GQA   6.29 MB   saving 4x (18.87 MB freed)
  seq= 8192:  MHA  100.66 MB   GQA  25.17 MB   saving 4x (75.50 MB freed)

K+V projection params:  MHA 524,288   GQA 131,072   (saved 393,216)


In [ ]:
# Cell 6 — Assemble the LLaMA-style block

class LlamaBlock(nn.Module):
    def __init__(self, cfg, cos, sin):
        super().__init__()
        d = cfg["d_model"]
        d_ff_swiglu = round((2/3) * cfg["d_ff"])     # 1365 — matched width from Cell 3
        self.norm1 = RMSNorm(d)
        self.attn  = GQAttention(cfg, cos, sin)
        self.norm2 = RMSNorm(d)
        self.ffn   = SwiGLU(d, d_ff_swiglu)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # pre-norm attention sublayer
        x = x + self.ffn(self.norm2(x))    # pre-norm FFN sublayer
        return x


block = LlamaBlock(cfg, cos, sin).to(device)
xb = torch.randn(2, 32, cfg["d_model"], device=device)
out = block(xb)
print(f"Forward OK — in {tuple(xb.shape)} -> out {tuple(out.shape)}")

# Per-component param breakdown
def n_params(m): return sum(p.numel() for p in m.parameters())
print(f"\nLlamaBlock param breakdown:")
print(f"  norm1 (RMSNorm):   {n_params(block.norm1):>10,}")
print(f"  attn  (GQA+RoPE):  {n_params(block.attn):>10,}")
print(f"  norm2 (RMSNorm):   {n_params(block.norm2):>10,}")
print(f"  ffn   (SwiGLU):    {n_params(block.ffn):>10,}")
print(f"  ----------------------------------")
print(f"  total per block:   {n_params(block):>10,}")

# Residual sanity: a freshly-initialized pre-norm block should be near-identity-ish
# (SwiGLU down-proj and attn out-proj start small relative to the residual)
delta = (out - xb).abs().mean().item()
sig   = xb.abs().mean().item()
print(f"\nResidual check — mean|out-in|={delta:.4f} vs mean|in|={sig:.4f} "
      f"(ratio {delta/sig:.2f})")

Forward OK — in (2, 32, 512) -> out (2, 32, 512)

LlamaBlock param breakdown:
  norm1 (RMSNorm):          512
  attn  (GQA+RoPE):     655,360
  norm2 (RMSNorm):          512
  ffn   (SwiGLU):     2,096,640
  ----------------------------------
  total per block:    2,753,024

Residual check — mean|out-in|=0.1272 vs mean|in|=0.7939 (ratio 0.16)


In [ ]:
# Cell 7 — Build GPT (faithful notebook-04 rebuild) and LLaMA models, compare statically

# --- GPT-side components (standard, WITH biases, to match the original 44.78M model) ---
class MHAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_h = cfg["num_heads"]; self.hd = cfg["d_model"] // self.n_h
        d = cfg["d_model"]
        self.w_q = nn.Linear(d, d); self.w_k = nn.Linear(d, d)
        self.w_v = nn.Linear(d, d); self.w_o = nn.Linear(d, d)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.w_q(x).view(B, T, self.n_h, self.hd).transpose(1, 2)
        k = self.w_k(x).view(B, T, self.n_h, self.hd).transpose(1, 2)
        v = self.w_v(x).view(B, T, self.n_h, self.hd).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.w_o(out.transpose(1, 2).contiguous().view(B, T, self.n_h * self.hd))

class GELU_FFN_bias(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d, d_ff); self.fc2 = nn.Linear(d_ff, d)
    def forward(self, x): return self.fc2(F.gelu(self.fc1(x)))

class GPTBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.norm1 = nn.LayerNorm(d); self.attn = MHAttention(cfg)
        self.norm2 = nn.LayerNorm(d); self.ffn  = GELU_FFN_bias(d, cfg["d_ff"])
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

def _init(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, std=0.02)

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["max_len"],   cfg["d_model"])   # learned absolute PE
        self.blocks  = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg["num_layers"])])
        self.norm_f  = nn.LayerNorm(cfg["d_model"])
        self.head    = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight                          # weight tying
        self.apply(_init)
    def forward(self, idx):
        T = idx.shape[1]
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm_f(x))

class LlamaModel(nn.Module):
    def __init__(self, cfg, cos, sin):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])  # no pos table — RoPE
        self.blocks  = nn.ModuleList([LlamaBlock(cfg, cos, sin) for _ in range(cfg["num_layers"])])
        self.norm_f  = RMSNorm(cfg["d_model"])
        self.head    = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight                          # weight tying
        self.apply(_init)
    def forward(self, idx):
        x = self.tok_emb(idx)                                           # position enters via RoPE
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm_f(x))


torch.manual_seed(42)
gpt   = GPTModel(cfg).to(device)
llama = LlamaModel(cfg, cos, sin).to(device)

def n_params(m): return sum(p.numel() for p in m.parameters())
gp, lp = n_params(gpt), n_params(llama)
print(f"GPT   (notebook-04 rebuild): {gp:,} params")
print(f"LLaMA (modern variant):      {lp:,} params")
print(f"Difference: {gp - lp:,}  ({(lp/gp - 1)*100:+.1f}%)\n")

# Forward both + random-init loss baseline (should be ~ ln(vocab), the notebook-05 lesson)
idx = torch.randint(0, cfg["vocab_size"], (2, 32), device=device)
tgt = torch.randint(0, cfg["vocab_size"], (2, 32), device=device)
for name, m in [("GPT", gpt), ("LLaMA", llama)]:
    with torch.no_grad():
        logits = m(idx)
        loss = F.cross_entropy(logits.view(-1, cfg["vocab_size"]), tgt.view(-1))
    print(f"{name:5} forward {tuple(idx.shape)} -> {tuple(logits.shape)} | "
          f"init loss {loss.item():.3f}")
print(f"\nTheoretical random-init baseline ln(vocab) = {math.log(cfg['vocab_size']):.3f}")

GPT   (notebook-04 rebuild): 44,777,984 params
LLaMA (modern variant):      42,250,240 params
Difference: 2,527,744  (-5.6%)

GPT   forward (2, 32) -> (2, 32, 50257) | init loss 10.923
LLaMA forward (2, 32) -> (2, 32, 50257) | init loss 10.915

Theoretical random-init baseline ln(vocab) = 10.825


In [ ]:
# Cell 8 — Train GPT vs LLaMA on the lighthouse corpus (matched optimizer & budget)
import tiktoken

enc = tiktoken.get_encoding("gpt2")

corpus = """
The old lighthouse stood at the edge of the cliff, its light sweeping
across the dark water every few seconds. Sailors who passed that coast
said the keeper never slept, that he watched the horizon for ships that
would never come. Every winter the storms grew fiercer, and every winter
the lighthouse held its ground. The keeper would climb the spiral stairs
each evening, counting them as he always had, and light the lamp before
the last color drained from the sky. He believed that somewhere out on
the water, someone was always looking back toward the light, grateful
for the proof that land still existed, that the world had not been
swallowed entirely by the sea.
"""

ids = torch.tensor(enc.encode(corpus), device=device)
seq_len, stride = 10, 5
X, Y = [], []
for i in range(0, len(ids) - seq_len, stride):
    X.append(ids[i:i+seq_len]); Y.append(ids[i+1:i+seq_len+1])
X, Y = torch.stack(X), torch.stack(Y)
print(f"Corpus: {len(ids)} tokens -> {len(X)} training examples\n")

def train(model, name, epochs=100):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    model.train()
    for ep in range(epochs):
        opt.zero_grad()
        logits = model(X)
        loss = F.cross_entropy(logits.view(-1, cfg["vocab_size"]), Y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if ep == 0 or (ep+1) % 20 == 0:
            print(f"  {name:5} epoch {ep+1:>3}: loss {loss.item():.4f}")
    return loss.item()

torch.manual_seed(42); gpt_final   = train(gpt,   "GPT")
print()
torch.manual_seed(42); llama_final = train(llama, "LLaMA")

print(f"\nFinal loss — GPT: {gpt_final:.4f} | LLaMA: {llama_final:.4f}")
print(f"Random-init baseline: {math.log(cfg['vocab_size']):.3f}")

Corpus: 146 tokens -> 28 training examples

  GPT   epoch   1: loss 10.9871
  GPT   epoch  20: loss 1.7372
  GPT   epoch  40: loss 0.0651
  GPT   epoch  60: loss 0.0306
  GPT   epoch  80: loss 0.0271
  GPT   epoch 100: loss 0.0262

  LLaMA epoch   1: loss 10.9029
  LLaMA epoch  20: loss 0.7744
  LLaMA epoch  40: loss 0.0603
  LLaMA epoch  60: loss 0.0321
  LLaMA epoch  80: loss 0.0282
  LLaMA epoch 100: loss 0.0267

Final loss — GPT: 0.0262 | LLaMA: 0.0267
Random-init baseline: 10.825


In [ ]:
# Cell 9 — Notebook 07 summary

print("=" * 60)
print("NOTEBOOK 07: MODERN ARCHITECTURE VARIANTS (LLaMA-style)")
print("=" * 60)

print("\n✅ Components built and verified:")
print("  1. RMSNorm   — scale-only norm; == biasless LayerNorm on zero-mean input")
print("  2. SwiGLU    — gated FFN; 2/3 width trick holds params at 1.000x GELU")
print("  3. RoPE      — rotary PE; ⟨R(m)q,R(n)k⟩ proven to depend only on (n−m)")
print("  4. GQA       — 8 query / 2 KV heads; 4x smaller KV cache, norm-preserving")
print("  +  Flash attn (conceptual) — exact attention via SDPA's fused backend")

print("\n🔑 Key concepts:")
print("  • RMSNorm fixes the 2nd moment, leaves the 1st (mean) free")
print("  • RoPE is orthogonal -> norm-preserving; position lives in phase")
print("  • RoPE decay = frequency-bank dephasing (non-monotonic, by design)")
print("  • GQA saves on K/V only; query side keeps full expressivity")
print("  • Flash attention: same math, O(N) memory via tiling + online softmax")
print("  • std=0.02 + weight tying held logit variance through the full rewrite")

print("\n📊 A/B vs. notebook-04 GPT (exact rebuild):")
print(f"  • GPT params:   {gp:,}")
print(f"  • LLaMA params: {lp:,}  ({(lp/gp-1)*100:+.1f}% — leaner, as a side effect)")
print(f"  • Final loss — GPT {gpt_final:.4f} | LLaMA {llama_final:.4f} (matched)")
print(f"  • Both init losses ≈ ln(vocab) = {math.log(cfg['vocab_size']):.3f}")

print("\n🎯 Next session — Days 13–14: Efficient Training Techniques")
print("    - Mixed precision (fp16/bf16)")
print("    - Gradient checkpointing")
print("    - KV-cache for inference (GQA pays off here)")
print("    - Memory optimization for the T4")

print("\n" + "=" * 60)
print("12/21 days complete (~57%) — save this notebook!")
print("=" * 60)

NOTEBOOK 07: MODERN ARCHITECTURE VARIANTS (LLaMA-style)

✅ Components built and verified:
  1. RMSNorm   — scale-only norm; == biasless LayerNorm on zero-mean input
  2. SwiGLU    — gated FFN; 2/3 width trick holds params at 1.000x GELU
  3. RoPE      — rotary PE; ⟨R(m)q,R(n)k⟩ proven to depend only on (n−m)
  4. GQA       — 8 query / 2 KV heads; 4x smaller KV cache, norm-preserving
  +  Flash attn (conceptual) — exact attention via SDPA's fused backend

🔑 Key concepts:
  • RMSNorm fixes the 2nd moment, leaves the 1st (mean) free
  • RoPE is orthogonal -> norm-preserving; position lives in phase
  • RoPE decay = frequency-bank dephasing (non-monotonic, by design)
  • GQA saves on K/V only; query side keeps full expressivity
  • Flash attention: same math, O(N) memory via tiling + online softmax
  • std=0.02 + weight tying held logit variance through the full rewrite

📊 A/B vs. notebook-04 GPT (exact rebuild):
  • GPT params:   44,777,984
  • LLaMA params: 42,250,240  (-5.6% — leaner, a